In [ ]:
import re
from copy import deepcopy
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from Bio import Phylo

OUT = Path("results/comparison")
OUT.mkdir(parents=True, exist_ok=True)
FIG = Path("figures")
FIG.mkdir(parents=True, exist_ok=True)


def norm(text):
    text = str(text).lower().replace("_", "-")
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-")


def contains_token(haystack, needle):
    if len(needle) < 4:
        return False
    return f"-{needle}-" in f"-{haystack}-"


def assembly_of(name):
    return str(name).split("|", 1)[0]


def support_of(clade):
    text = str(clade.name) if clade.name else ""
    if "/" not in text:
        return None
    left, right = text.split("/", 1)
    try:
        return float(left), float(right)
    except ValueError:
        return None


def unrooted_splits(tree):
    universe = {tip.name for tip in tree.get_terminals()}
    found = set()
    for clade in tree.get_nonterminals():
        members = frozenset(tip.name for tip in clade.get_terminals())
        if 1 < len(members) < len(universe):
            other = frozenset(universe - members)
            found.add(members if len(members) <= len(other) else other)
    return found


def prune_to(tree, keep):
    for tip in list(tree.get_terminals()):
        if tip.name not in keep:
            tree.prune(tip)
    changed = True
    while changed:
        changed = False
        for clade in list(tree.find_clades()):
            if clade is tree.root or clade.is_terminal() or len(clade.clades) != 1:
                continue
            child = clade.clades[0]
            child.branch_length = float(child.branch_length or 0) + float(clade.branch_length or 0)
            parent = next(candidate for candidate in tree.find_clades() if clade in candidate.clades)
            parent.clades[parent.clades.index(clade)] = child
            changed = True
            break
    return tree


cross = pd.read_csv("data/gentilhomme/gentilhomme_ncbi_crosswalk.tsv", sep="\t", dtype=str).fillna("")
patric = pd.read_csv("data/gentilhomme/patric_rooted.tsv", sep="\t", dtype=str).fillna("")
patric_name = dict(zip(patric["genome.genome_id"], patric["genome.genome_name"]))
cross_row = {row.patric_genome_id: row for row in cross.itertuples(index=False)}
meta = pd.read_csv("results/sequences/colap_clean_metadata.csv", dtype=str).fillna("")
status = pd.read_csv("results/sequences/genome_status.csv", dtype=str).fillna("")
clean_by_assembly = meta.set_index("assembly_accession")
status_by_assembly = status.set_index("assembly_accession")
clean_by_group = {group: frame for group, frame in meta.groupby("strain_group")}
species = Phylo.read("data/gentilhomme/species_tree.nwk", "newick")
colap = Phylo.read("results/phylogeny/colap_ml.treefile", "newick")
colap_by_assembly = {assembly_of(tip.name): tip.name for tip in colap.get_terminals()}
rows = []
for tip in species.get_terminals():
    patric_id = tip.name
    info = cross_row.get(patric_id)
    name = patric_name.get(patric_id, getattr(info, "patric_name", "") if info else "")
    method = info.match_method if info is not None else ""
    accession = info.ncbi_representative_accession if info is not None else ""
    match_assembly = ""
    match_status = "species_only"
    note = ""
    if "pseudoalteromonas" in name.lower() or name.lower().startswith("alteromonas"):
        match_status = "outgroup_not_in_colap"
        note = "phylogenomic outgroup, absent from the ColAP sample"
    elif accession and accession in clean_by_assembly.index:
        match_assembly = accession
        match_status = "matched"
        note = f"crosswalk assembly, method {method}"
    elif accession and accession in status_by_assembly.index:
        group = status_by_assembly.loc[accession, "strain_group"]
        hits = clean_by_group.get(group, pd.DataFrame())
        if len(hits) == 1:
            match_assembly = hits.iloc[0].assembly_accession
            match_status = "matched"
            note = f"another assembly of the same strain was kept: {match_assembly}"
        elif len(hits) > 1:
            match_status = "ambiguous"
            note = f"several clean assemblies in {group}"
        else:
            note = f"{accession} is in the genome table but not in the clean ColAP set"
    else:
        haystack = norm(name)
        hits = []
        for strain_row in meta.itertuples(index=False):
            token = norm(strain_row.strain)
            organism = norm(strain_row.organism)
            if contains_token(haystack, token) or (token and token == organism):
                hits.append(strain_row.assembly_accession)
        hits = sorted(set(hits))
        if len(hits) == 1:
            match_assembly = hits[0]
            match_status = "matched"
            method = method or "strain_token_patric_table"
            note = "unique strain token from the PATRIC name"
        elif len(hits) > 1:
            match_status = "ambiguous"
            note = "PATRIC strain matched several ColAP rows"
        else:
            note = "no unique assembly or strain match"
    genus = organism = strain = ""
    if match_assembly:
        chosen = clean_by_assembly.loc[match_assembly]
        genus = chosen.genus
        organism = chosen.organism
        strain = chosen.strain
    rows.append({
        "patric_genome_id": patric_id,
        "patric_name": name,
        "crosswalk_method": method,
        "crosswalk_assembly": accession,
        "colap_assembly": match_assembly,
        "colap_organism": organism,
        "colap_strain": strain,
        "colap_genus": genus,
        "match_status": match_status,
        "note": note,
    })
matched = pd.DataFrame(rows)
seen = {}
for index, row in matched.iterrows():
    assembly = row.colap_assembly
    if row.match_status != "matched" or not assembly:
        continue
    if assembly in seen:
        matched.at[index, "match_status"] = "duplicate_same_colap_strain"
        matched.at[index, "note"] = f"this ColAP strain is already matched to {seen[assembly]}"
        matched.at[index, "colap_assembly"] = ""
    else:
        seen[assembly] = row.patric_genome_id
matched_assemblies = set(matched.loc[matched.match_status == "matched", "colap_assembly"])
extra = []
for strain_row in meta.itertuples(index=False):
    if strain_row.assembly_accession in matched_assemblies:
        continue
    extra.append({
        "patric_genome_id": "",
        "patric_name": "",
        "crosswalk_method": "",
        "crosswalk_assembly": "",
        "colap_assembly": strain_row.assembly_accession,
        "colap_organism": strain_row.organism,
        "colap_strain": strain_row.strain,
        "colap_genus": strain_row.genus,
        "match_status": "colap_only",
        "note": "present in the ColAP tree and absent from the species tree",
    })
matched = pd.concat([matched, pd.DataFrame(extra)], ignore_index=True)
matched.to_csv(OUT / "matched_taxa.csv", index=False)
pairs = matched[matched.match_status == "matched"].copy()
keep_species = set(pairs.patric_genome_id)
keep_colap = {colap_by_assembly[assembly] for assembly in pairs.colap_assembly}
species_pruned = prune_to(Phylo.read("data/gentilhomme/species_tree.nwk", "newick"), keep_species)
colap_pruned = prune_to(Phylo.read("results/phylogeny/colap_ml.treefile", "newick"), keep_colap)
rename_species = dict(zip(pairs.patric_genome_id, pairs.colap_assembly))
rename_colap = {colap_by_assembly[assembly]: assembly for assembly in pairs.colap_assembly}
for tip in species_pruned.get_terminals():
    tip.name = rename_species[tip.name]
for tip in colap_pruned.get_terminals():
    tip.name = rename_colap[tip.name]
Phylo.write(species_pruned, OUT / "species_pruned.nwk", "newick")
Phylo.write(colap_pruned, OUT / "colap_pruned.nwk", "newick")
species_splits = unrooted_splits(species_pruned)
colap_splits = unrooted_splits(colap_pruned)
rf = len(species_splits - colap_splits) + len(colap_splits - species_splits)
n_taxa = len(pairs)
max_rf = 2 * (n_taxa - 3)
shared = set(pairs.colap_assembly)
induced = {}
for clade in colap.get_nonterminals():
    pair = support_of(clade)
    if pair is None or pair[0] < 80 or pair[1] < 95:
        continue
    members = {assembly_of(tip.name) for tip in clade.get_terminals()} & shared
    if 1 < len(members) < len(shared) - 1:
        other = shared - members
        split = frozenset(members if len(members) <= len(other) else other)
        previous = induced.get(split)
        if previous is None or pair[1] > previous[1] or (pair[1] == previous[1] and pair[0] > previous[0]):
            induced[split] = (pair[0], pair[1], len(members))
concordant = [split for split in induced if split in species_splits]
conflicting = [split for split in induced if split not in species_splits]
meta_by = meta.set_index("assembly_accession")
conflict_rows = []
for split in sorted(conflicting, key=lambda item: (-induced[item][1], len(item))):
    alrt, ufboot, n_full = induced[split]
    genera = meta_by.loc[list(split), "genus"].value_counts().to_dict()
    labels = [f"{meta_by.loc[acc, 'genus']} {meta_by.loc[acc, 'strain']}" for acc in sorted(split)]
    conflict_rows.append({
        "small_side_n": len(split),
        "full_clade_shared_n": n_full,
        "sh_alrt": alrt,
        "ufboot": ufboot,
        "genera": "; ".join(f"{genus}:{count}" for genus, count in sorted(genera.items())),
        "taxa": "; ".join(labels),
        "in_species_tree": False,
    })
pd.DataFrame(conflict_rows).to_csv(OUT / "supported_conflicts.csv", index=False)
stats = pd.DataFrame([{
    "species_tips": len({tip.name for tip in species.get_terminals()}),
    "colap_tips": len(colap_by_assembly),
    "matched": int((matched.match_status == "matched").sum()),
    "colap_only": int((matched.match_status == "colap_only").sum()),
    "species_only": int((matched.match_status == "species_only").sum()),
    "outgroup_not_in_colap": int((matched.match_status == "outgroup_not_in_colap").sum()),
    "ambiguous": int((matched.match_status == "ambiguous").sum()),
    "duplicate_same_colap_strain": int((matched.match_status == "duplicate_same_colap_strain").sum()),
    "robinson_foulds": rf,
    "max_robinson_foulds": max_rf,
    "normalized_robinson_foulds": rf / max_rf,
    "supported_induced_splits": len(induced),
    "supported_concordant": len(concordant),
    "supported_conflicting": len(conflicting),
    "species_tree_branch_support": "none in Newick",
}])
stats.to_csv(OUT / "tree_comparison_statistics.csv", index=False)
display = deepcopy(colap)
for tip in display.get_terminals():
    tip.name = assembly_of(tip.name)
display.root_at_midpoint()
display.ladderize()
protected = {
    "GCF_000012325.1", "GCF_002836245.1", "GCF_000764225.1", "GCF_007954275.1",
    "GCF_007097525.1", "GCA_002162925.1", "GCF_000764205.1", "GCA_000712155.1", "GCA_002734005.2",
}
chosen = []
for clade in display.get_nonterminals():
    if clade is display.root:
        continue
    pair = support_of(clade)
    names = [tip.name for tip in clade.get_terminals()]
    if pair is None or pair[0] < 80 or pair[1] < 95 or len(names) < 8 or protected.intersection(names):
        continue
    chosen.append(clade)
chosen.sort(key=lambda clade: len(clade.get_terminals()), reverse=True)
covered = set()
collapsed = []
for clade in chosen:
    names = {tip.name for tip in clade.get_terminals()}
    if names & covered:
        continue
    collapsed.append(clade)
    covered |= names
for clade in collapsed:
    names = [tip.name for tip in clade.get_terminals()]
    genera = meta_by.loc[names, "genus"].value_counts()
    label = ", ".join(f"{genus} {count}" for genus, count in genera.items())
    pair = support_of(clade)
    clade.clades = []
    clade.name = f"{label} (n={len(names)}; {pair[0]:.1f}/{pair[1]:.0f})"
represented = 0
for tip in display.get_terminals():
    if tip.name.startswith("GCF_") or tip.name.startswith("GCA_"):
        represented += 1
    else:
        represented += int(tip.name.split("n=", 1)[1].split(";", 1)[0])
if represented != len(colap_by_assembly):
    raise SystemExit(f"collapsed tree represents {represented} strains")


def place(clade, cursor, x):
    clade.x = x
    if not clade.clades:
        clade.y = cursor[0]
        cursor[0] += 1
        return
    for child in clade.clades:
        place(child, cursor, x + float(child.branch_length or 0))
    clade.y = sum(child.y for child in clade.clades) / len(clade.clades)


place(display.root, [0], 0.0)
plt.rcParams.update({"font.family": "DejaVu Sans"})
figure, axis = plt.subplots(figsize=(11.2, 10.4))
def draw(clade):
    for child in clade.clades:
        axis.plot([clade.x, clade.x], [clade.y, child.y], color="#444444", lw=0.7)
        axis.plot([clade.x, child.x], [child.y, child.y], color="#444444", lw=0.7)
        pair = support_of(child)
        if pair is not None and pair[0] >= 80 and pair[1] >= 95:
            axis.scatter([child.x], [child.y], s=16, c="black", zorder=3)
        draw(child)
draw(display.root)
for tip in display.get_terminals():
    axis.text(tip.x + 0.01, tip.y, tip.name, va="center", ha="left", fontsize=7)
axis.set_axis_off()
figure.tight_layout()
figure.savefig(FIG / "colap_compact_tree.png", dpi=300)
plt.close()
